# Playground 05 — MVC: what goes where (and the swap trick)

📖 Primer: [docs/00-concepts.md](../docs/00-concepts.md), section 4

A complete Model-View-Controller app in three short cells — same layers as the real project, miniature scale. The grand finale swaps out the model's storage WITHOUT touching the controller, which is the whole reason MVC exists. Watch for it.

## MODEL — what the data *is* and what it *can do*

(real: `app/models.py`, `app/store.py`) — knows nothing about requests, JSON, or the web.

In [ ]:
class Bag:
    def __init__(self, id, brand, model, prices):
        self.id = id
        self.brand = brand
        self.model = model
        self.prices = prices                # full price history

    def current_price(self):
        return self.prices[-1]              # most recent observation


class DictStore:
    # Keeps bags in a dict — like the real Phase 2 BagStore.
    def __init__(self):
        self._bags = {}

    def add(self, bag):
        self._bags[bag.id] = bag

    def cheapest(self, n):
        ranked = sorted(self._bags.values(), key=lambda b: b.current_price())
        return ranked[:n]

## VIEW — the shape the outside world sees

(real: `app/schemas.py`) — picks WHICH fields go out and WHAT they're named. No logic.

In [ ]:
def bag_response(bag):
    return {
        "id": bag.id,
        "label": f"{bag.brand} {bag.model}",          # renamed + combined!
        "current_price": bag.current_price(),
        # note what we DON'T send: the full price history stays private
    }

## CONTROLLER — the traffic cop

(real: the route functions in `app/main.py`) — calls the model, hands the result to the view. Notice: no business logic lives here. It only coordinates.

In [ ]:
def get_cheapest_bags(store, n):
    # Handles 'GET /bags/cheapest?n=...'
    bags = store.cheapest(n)                    # 1. ask the MODEL
    return 200, [bag_response(b) for b in bags] # 2. shape with the VIEW

## Showtime

In [ ]:
def fill(store):
    store.add(Bag("chanel-flap-001", "Chanel", "Classic Flap", [9500.0, 9800.0]))
    store.add(Bag("lv-neverfull-001", "Louis Vuitton", "Neverfull MM", [1800.0]))
    store.add(Bag("hermes-birkin-001", "Hermès", "Birkin 30", [22000.0, 23500.0]))


store = DictStore()
fill(store)
status, body = get_cheapest_bags(store, n=2)
print(f"GET /bags/cheapest?n=2  ->  {status}")
for item in body:
    print(f"  {item}")

## The swap — a totally different storage, SAME controller

In [ ]:
class ListStore:
    # Stores bags in a plain list instead of a dict. The internals are
    # different — but it offers the SAME two methods, so nobody
    # upstairs can tell the difference. In Phase 5.5 the 'different
    # internals' will be an entire PostgreSQL database.
    def __init__(self):
        self._bags = []

    def add(self, bag):
        self._bags.append(bag)

    def cheapest(self, n):
        return sorted(self._bags, key=lambda b: b.current_price())[:n]


store = ListStore()                            # <-- the ONLY changed line
fill(store)
status, body = get_cheapest_bags(store, n=2)   # untouched controller!
print(f"GET /bags/cheapest?n=2  ->  {status}")
for item in body:
    print(f"  {item}")

Identical output, zero controller edits. **That** is MVC's payoff, and you'll feel it for real in Phase 5.5 (`BagStore` → `SqlBagStore`).

## ✏️ Your turn

In [ ]:
# Exercise 1 — VIEW change: edit bag_response (scroll up, edit, re-run
# that cell) so it ALSO sends "brand" as its own field. Then re-run
# the Showtime cell. Which layer did you touch? Did the model or
# controller care?
print("edit the VIEW cell above, then re-run Showtime")

In [ ]:
# Exercise 2 — cause the primer's question-5 bug ON PURPOSE: make the
# view return current_price as a dict {"price": ..., "currency": "EUR"}.
# Notice you broke the OUTPUT without touching any logic — that's why
# 'wrong label, right numbers' points at the view.
print("edit the VIEW cell above, re-run it + Showtime, observe the EUR bug")

In [ ]:
# Exercise 3 — MODEL change: add a method to Bag (scroll up, edit,
# re-run that cell, then re-run Showtime):
#
#     def price_change(self):
#         return self.prices[-1] - self.prices[0]
#
# ...and add it to the view's output. (This is literally Phase 1's
# exercise — sshh.)
print("edit the MODEL + VIEW cells above")

In [ ]:
# Exercise 4 — CONTROLLER guardrail: edit get_cheapest_bags so that
# if n <= 0, it returns (400, {"error": "n must be positive"}).
# Input-checking IS a controller job — now each layer has earned
# its keep. Then check it works:
status, body = get_cheapest_bags(store, n=-3)
print(status, body)   # want: 400 {'error': 'n must be positive'}